In [1]:
import os
import csv
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
from PIL import Image
from tqdm import tqdm

import random

seed = 42

random.seed(seed)                  # Python built-in random
np.random.seed(seed)               # NumPy
torch.manual_seed(seed)            # PyTorch (CPU)
torch.cuda.manual_seed(seed)       # PyTorch (single GPU)
torch.cuda.manual_seed_all(seed)   # PyTorch (all GPUs)

# Ensures deterministic behavior
# torch.backends.cudnn.deterministic = True
# torch.backends.cudnn.benchmark = False

In [2]:
#-------------读取训练集,训练集地址已经设定好，下面这段不用修改------------------#
#-----Read the training set, the address of the training set has been set, and the following section does not need to be modified-------#
train_path = "/bohr/train-i5ob/v1"

In [15]:
# 读取数据。
def load_train_data(data_dir='./train/'):
    label_path = os.path.join(data_dir, 'train_labels.csv')
    image_dir = os.path.join(data_dir, 'train_images')

    data = []
    with open(label_path, 'r', newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            sample_id = row['id']
            data.append({
                'id': sample_id,
                'image_path': os.path.join(image_dir, f'{sample_id}.png'),
                'sum': float(row['sum']),
                'product': float(row['product']),
            })

    print(f'Successfully loaded training records: {len(data)}')
    return data

class MNISTBaselineDataset(Dataset):
    def __init__(self, records):
        self.records = records

    def __len__(self):
        return len(self.records)

    @staticmethod
    def load_image(image_path):
        image = Image.open(image_path).convert('L')
        image = np.array(image, dtype=np.float32) / 255.0
        return image

    def __getitem__(self, idx):
        record = self.records[idx]
        image = self.load_image(record['image_path'])
        image = torch.tensor(image, dtype=torch.float32).unsqueeze(0)
        target = torch.tensor([record['sum'], record['product']], dtype=torch.float32)
        return image, target


def create_train_loader(batch_size=64, num_workers=4, data_dir='./train/'):
    records = load_train_data(data_dir)
    dataset = MNISTBaselineDataset(records)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=True
    )

import torch.nn.functional as F
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)  # 输入通道1，输出通道32
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1) # 输入通道32，输出通道64
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)        # 最大池化层
        self.fc1 = nn.Linear(64 * 7 * 7, 128)                   # 全连接层
        self.fc2 = nn.Linear(128, 10)                            # 输出层，10个类别

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # 卷积 + ReLU + 池化
        x = self.pool(F.relu(self.conv2(x)))  # 卷积 + ReLU + 池化
        x = x.view(-1, 64 * 7 * 7)             # 展平
        x = F.relu(self.fc1(x))                # 全连接 + ReLU
        x = self.fc2(x)                        # 输出层
        return x

class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = SimpleCNN()

    def get_logits(self, x):
        # [B, 1, 28, 28*4] -> [B*4, 1, 28, 28]
        x = x.unfold(3, 28, 28).permute(0, 3, 1, 2, 4).reshape(-1, 1, 28, 28)
        # [B*4, 10]
        x = self.backbone(x)
        return F.softmax(x, dim=1)

    def forward(self, x):
        x = self.get_logits(x)
        # [B*4]
        x = (x * torch.arange(10, dtype=x.dtype, device=x.device)).sum(dim=1)
        # [B, 4] -> 2*[B, 1] -> [B, 2] 
        x = x.reshape(-1, 4)
        return torch.stack(
            (x.sum(dim=1), x.prod(dim=1)),
            dim=1
        )

    def predict(self, x):
        x = self.get_logits(x)
        # [B*4]
        x = torch.argmax(x, dim=1)
        # [B, 4] -> 2*[B, 1] -> [B, 2]
        x = x.reshape(-1, 4)
        return torch.stack(
            (x.sum(dim=1), x.prod(dim=1)),
            dim=1
        )


# 训练模型
def train(model, train_loader, epochs, device='cpu'):
    model.to(device)
    criterion = nn.HuberLoss()
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=2e-3, fused=True)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs*len(train_loader))

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        total_sum_loss = 0.0
        total_prod_loss = 0.0

        for images, targets in tqdm(train_loader, desc=f'Epoch {epoch}/{epochs}', leave=False):
            images = images.to(device)
            targets = targets.to(device)

            optimizer.zero_grad()
            preds = model(images)
            sum_loss = criterion(preds[:, 0], targets[:, 0])
            prod_loss = criterion(preds[:, 1], targets[:, 1]) * 0.015
            loss = sum_loss + prod_loss
            loss.backward()
            optimizer.step()
            scheduler.step()
            total_loss += loss.item() * images.size(0)
            total_sum_loss += sum_loss.item() * images.size(0)
            total_prod_loss += prod_loss.item() * images.size(0)

        avg_loss = total_loss / len(train_loader.dataset)
        avg_sum_loss = total_sum_loss / len(train_loader.dataset)
        avg_prod_loss = total_prod_loss / len(train_loader.dataset)
        print(f'Epoch {epoch}/{epochs} - Loss: {avg_loss:.4f}, Sum Loss: {avg_sum_loss:.4f}, Prod Loss: {avg_prod_loss:.4f}')

        if epoch % 5 == 0:
            # evaluate train acc
            with torch.inference_mode():
                correct = 0
                total = 0
                for images, targets in train_loader:
                    total += 2 * images.size(0)
                    images = images.to(device)
                    targets = targets.to(device)
                    # preds = model.predict(images)
                    preds = model(images).round()
                    correct += (preds == targets).to(torch.long).sum().item()
                print(f'Train Acc: {correct / total}')


def set_random_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


device = 'cuda' if torch.cuda.is_available() else 'cpu'
seed = 42
batch_size = 64
epochs = 400

set_random_seed(seed)
train_loader = create_train_loader(batch_size=batch_size, data_dir=train_path)
model = Model()
train(model, train_loader, epochs=epochs, device=str(device))

In [4]:
#-------------读取测试集---------------#“DATA_PATH”是测试集加密后的环境变量，按照如下方式可以在提交后，系统评分时访问测试集，但是选手无法直接下载
#----Read the testing set, “DATA_PATH” is an environment variable for the encrypted test set. After submission, you can access the test set for system scoring in the following manner, but the contestant cannot download it directly.-----#
if os.environ.get('DATA_PATH'):
    test_path = os.environ.get("DATA_PATH") + "/"
else:
    test_path = "./test/"
    print("Baseline 运行时，因为无法读取测试集，所以会有此条报错，属于正常现象")
    print("When baseline is running, this error message will appear because the test set cannot be read, which is a normal phenomenon.")
    #Baseline 运行时，因为无法读取测试集，所以会有此条报错，属于正常现象
    #When baseline is running, this error message will appear because the test set cannot be read, which is a normal phenomenon.

In [5]:
# 读取测试数据
class MNISTTestDataset(Dataset):
    def __init__(self, image_dir):
        self.image_paths = sorted(str(path) for path in Path(image_dir).glob('*.png'))

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        sample_id = Path(image_path).stem
        image = MNISTBaselineDataset.load_image(image_path)
        image = torch.tensor(image, dtype=torch.float32).unsqueeze(0)
        return image, sample_id


# 这里用训练好的模型直接回归 sum 和 product。
def predict_and_save(model, image_dir, output_file, device='cpu', batch_size=64, num_workers=1):
    dataset = MNISTTestDataset(image_dir)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
    )

    rows = []
    model.eval()
    with torch.inference_mode():
        for images, sample_ids in tqdm(loader, desc=f'Predicting {os.path.basename(image_dir)}', leave=False):
            images = images.to(device)
            preds = model.predict(images).cpu()
            sum_preds = preds[:, 0].numpy()
            product_preds = preds[:, 1].numpy()

            for sample_id, sum_pred, product_pred in zip(sample_ids, sum_preds, product_preds):
                rows.append({
                    'id': sample_id,
                    'sum': int(sum_pred),
                    'product': int(product_pred),
                })

    with open(output_file, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['id', 'sum', 'product'])
        writer.writeheader()
        writer.writerows(rows)

    print(f'Saved predictions to {output_file}')


val_dir = os.path.join(test_path, 'val')
test_dir = os.path.join(test_path, 'test')

for required_dir in [val_dir, test_dir]:
    if not os.path.isdir(required_dir):
        raise FileNotFoundError(f'Missing test directory: {required_dir}')

predict_and_save(model, val_dir, 'submission_val.csv', device=str(device), batch_size=batch_size)
predict_and_save(model, test_dir, 'submission_test.csv', device=str(device), batch_size=batch_size)

In [ ]:
import zipfile

# 定义要打包的文件和压缩文件名
files_to_zip = ['submission_val.csv', 'submission_test.csv']
zip_filename = 'submission.zip'

# 创建一个 zip 文件
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in files_to_zip:
        # 将文件添加到 zip 文件中
        zipf.write(file, os.path.basename(file))

print(f'{zip_filename} 创建成功!')